# Generating shrunken logFC (posterior-mean) matrices

Runs the ChemoGeneticScreens pipeline through `module_finder`, producing matrices that
carry **both** the full-data and the subsampled estimates as separate rows.

Pipeline (all four scripts vendored unchanged under `src/module_finder/de/`):

| step | script | what it does |
|---|---|---|
| 1 | `ComputeSE.py` | cells → per-label `mean_diff` + Welch `se`, chunked |
| 2 | `run_ashr_on_chunk.R` | `ash(mean_diff, se)` per chunk |
| 3 | `MergeAshrRes.py` | merge label/gene metadata back in |
| 4 | `AshrGenerateMatrix.py` | pivot to wide label × gene `PosteriorMean` |

The only added code is *input construction*: one label per intended output row
(`<pert>` for the full estimate, `<pert>__sub<k>` for a subsample), with subsampled
cells duplicated so a single `ComputeSE.py` pass yields both — and one ashr fit covers
them, which is where the differing precisions belong since `ash` conditions on `se`.

**Subsampling rule.** A perturbation is subsampled only if it has **more than 50 cells**
in that context; then `n` is drawn uniformly at random from `[50, n_cells]`, and the
cells themselves are drawn uniformly without replacement. Perturbations at or below
50 contribute their full row only.

**Control rule.** A fixed set of 10,000 control cells is drawn once per context and
shared by every row — full and subsampled alike. `ComputeSE.py` computes the control
mean/variance once before its per-label loop, so one control group necessarily serves
all labels; fixing its size makes the `var_c / n_ctrl` term of the standard error
identical across rows and comparable across contexts.

Requires R with `ashr` (installed here: 2.2.63).

In [ ]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parents[1]          # notebooks/module_finder -> project root
sys.path.insert(0, str(PROJECT / "src"))

import numpy as np
import pandas as pd
import anndata as ad

from module_finder.de import (
    generate_posterior_matrices,
    plan_augmentation,
    build_augmented_adata,
    MIN_CELLS_TO_SUBSAMPLE,
    N_CONTROL_CELLS,
)

OUT_DIR = PROJECT / "data" / "posterior_matrices"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("project :", PROJECT)
print("outputs :", OUT_DIR)
print("floor   :", MIN_CELLS_TO_SUBSAMPLE, "cells")
print("controls:", N_CONTROL_CELLS, "cells (fixed per context)")

## 1. Pick an input

One context per call. The ChemoGenetic per-drug files carry no drug column, so the
context name is passed in and used to name the outputs.

Sizes to be aware of before choosing:

- `/processed_datasets/VCI/ChemoGenetic_H1_Basak/<drug>/<drug>.h5ad` — **241 GB** per drug,
  2.70M cells × 18,154 genes. `ComputeSE.py` calls `read_h5ad` (not backed), so this needs
  a very large machine or the `SPLIT/` shards.
- `/processed_datasets/VCI/ChemoGenetic_H1_Basak_Pathways/<drug>.h5ad` — **12.6 GB** per drug,
  2.37M cells × 7,589 genes, and it *does* carry a `drug` column. Tractable.

Set `SUBSET_PERTS` to a small number for a quick pass; `None` uses everything.

In [ ]:
DRUG = "Stattic"
ADATA_PATH = Path(f"/processed_datasets/VCI/ChemoGenetic_H1_Basak_Pathways/{DRUG}.h5ad")
PERTURBATION_KEY = "target_gene"
CONTROL_LABEL = "non-targeting"

SUBSET_PERTS = 40      # None for the full screen
SUBSET_GENES = 1500    # None for all genes

adata = ad.read_h5ad(ADATA_PATH, backed="r")
print(adata)
counts = adata.obs[PERTURBATION_KEY].value_counts()
print(f"\nperturbations: {len(counts)}")
print(f"controls      : {counts.get(CONTROL_LABEL, 0):,} cells")
print(f"cells/pert    : min {counts.min()}, median {int(counts.median())}, max {counts.max():,}")
print(f"eligible for subsampling (>{MIN_CELLS_TO_SUBSAMPLE} cells): "
      f"{int((counts.drop(CONTROL_LABEL, errors='ignore') > MIN_CELLS_TO_SUBSAMPLE).sum())}")

In [ ]:
# Optional subset, loaded into memory. Skip this cell to run on everything.
if SUBSET_PERTS is not None:
    keep_perts = [CONTROL_LABEL] + [
        p for p in counts.index if p != CONTROL_LABEL
    ][:SUBSET_PERTS]
    mask = adata.obs[PERTURBATION_KEY].isin(keep_perts).to_numpy()
    genes = slice(None) if SUBSET_GENES is None else slice(0, SUBSET_GENES)
    adata = adata[mask, genes].to_memory()
    print(adata.shape, "|", adata.obs[PERTURBATION_KEY].nunique(), "labels")

## 2. Inspect the augmentation plan

`plan_augmentation` decides the rows before any expression is touched, so the rule is
auditable on its own. One row per eventual matrix row.

In [ ]:
perturbations = adata.obs[PERTURBATION_KEY].astype(str).to_numpy()

cell_index, labels, plan = plan_augmentation(
    perturbations,
    control_label=CONTROL_LABEL,
    n_subsamples=2,
    n_control_cells=N_CONTROL_CELLS,
    seed=0,
)

print(f"{len(plan)} planned rows from {adata.n_obs:,} cells "
      f"-> {len(cell_index):,} cells after duplication\n")
print(plan["variant"].value_counts().to_string())
plan.head(12)

In [ ]:
# The rule, checked directly against the plan
full = plan[plan.variant == "full"].set_index("perturbation")["n_cells_planned"]
sub = plan[plan.variant == "subsample"]

too_small = full[full <= MIN_CELLS_TO_SUBSAMPLE].index
print("perturbations at/below the floor:", len(too_small))
print("  none of them subsampled:", not sub.perturbation.isin(too_small).any())
print("  control set fixed at:",
      int(plan.loc[plan.variant == "control", "n_cells_planned"].item()), "cells")
print("  every n within [50, n_full]:",
      bool(((sub.n_cells_planned >= MIN_CELLS_TO_SUBSAMPLE) &
            (sub.n_cells_planned <= full.reindex(sub.perturbation).to_numpy())).all()))

ax = sub.assign(frac=sub.n_cells_planned / full.reindex(sub.perturbation).to_numpy()) \
        .frac.plot.hist(bins=25, title="subsample size as a fraction of the full set")
ax.set_xlabel("n_subsample / n_full");

## 3. Build the augmented AnnData

Full-data cells keep their perturbation label; subsampled cells are duplicated under
`<pert>__sub<k>`. Controls are **never** duplicated — the full and subsampled rows share
one baseline, exactly as in the original pipeline.

In [ ]:
augmented, plan = build_augmented_adata(
    adata,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    n_subsamples=2,
    n_control_cells=N_CONTROL_CELLS,
    seed=0,
)
print(f"{adata.n_obs:,} cells -> {augmented.n_obs:,} "
      f"({augmented.n_obs / adata.n_obs:.2f}x)")
augmented.obs["mf_label"].value_counts().head(8)

## 4. Run the pipeline

`generate_posterior_matrices` does steps 2–3 above and then shells out to
`RunAshrPipeline.sh`, so this one call covers the whole chain. It writes the augmented
`.h5ad`, runs it, and deletes it again unless `keep_augmented_h5ad=True`.

`chunk_perts` reproduces the original per-100-perturbation ashr grouping. Full and
subsampled rows of one perturbation can then land in different chunks and get different
priors — raise it above the row count for a single shared prior.

In [ ]:
run = generate_posterior_matrices(
    adata,
    context=DRUG,
    out_dir=OUT_DIR,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    n_subsamples=2,
    n_control_cells=N_CONTROL_CELLS,
    seed=0,
    min_cells=20,
    chunk_perts=100,
    n_ashr_jobs=6,
    matrix_format="csv",
)
print(run)
print("\nmatrix        :", run.matrix_path)
print("row metadata  :", run.row_metadata_path)
print("ashr version  :", run.meta["ashr_version"])
print("build %.1fs, pipeline %.1fs" % (run.meta["build_seconds"], run.meta["pipeline_seconds"]))

## 5. The matrix and its per-row cell counts

`run.matrix` is labels × genes: `<pert>` rows are full-data estimates, `<pert>__sub<k>`
rows are subsamples. `run.row_metadata` records how many cells produced each row —
`n_cells_used` comes from the `n_pert` column the pipeline itself wrote, not from a
recomputation here.

In [ ]:
print(run.matrix.shape)
run.matrix.iloc[:6, :5]

In [ ]:
run.row_metadata.head(10)

In [ ]:
meta = run.row_metadata.set_index("label")

# how much does an estimate move when it is built from fewer cells?
fulls = meta[meta.variant == "full"].index
subs = meta[meta.variant == "subsample"]
rows = []
for label, r in subs.iterrows():
    if r.perturbation in run.matrix.index:
        a = run.matrix.loc[r.perturbation]
        b = run.matrix.loc[label]
        rows.append({
            "perturbation": r.perturbation,
            "n_full": int(meta.loc[r.perturbation, "n_cells_used"]),
            "n_sub": int(r.n_cells_used),
            "frac_cells": r.n_cells_used / meta.loc[r.perturbation, "n_cells_used"],
            "corr_to_full": np.corrcoef(a, b)[0, 1],
            "shrinkage_ratio": np.abs(b).mean() / max(np.abs(a).mean(), 1e-12),
        })
concordance = pd.DataFrame(rows)
print(concordance[["frac_cells", "corr_to_full", "shrinkage_ratio"]].describe().round(3).to_string())
concordance.head()

In [ ]:
# fewer cells -> larger se -> ashr shrinks harder. This is the expected direction.
ax = concordance.plot.scatter("frac_cells", "shrinkage_ratio", alpha=.6,
                              title="subsampled rows are shrunk more, as they should be")
ax.set_xlabel("fraction of cells used"); ax.set_ylabel("|PosteriorMean| relative to full");

## 6. The label-permuted null

Same call with `permute=True`: perturbation labels are shuffled among the perturbed
cells, controls untouched, every label keeping its cell count.

Note what this is a null *for*. It destroys which perturbation does what, but the
average perturbation effect still leaks into every row equally, because the control
baseline is left intact. It is a null for **specificity**, not for overall effect
magnitude.

In [ ]:
null_run = generate_posterior_matrices(
    adata,
    context=DRUG,
    out_dir=OUT_DIR,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    n_subsamples=2,
    n_control_cells=N_CONTROL_CELLS,
    seed=0,
    permute=True,
    permute_seed=0,
    min_cells=20,
    chunk_perts=100,
    n_ashr_jobs=6,
)
print(null_run)

real_spread = run.matrix.std(axis=0).mean()
null_spread = null_run.matrix.std(axis=0).mean()
print(f"\nmean across-row SD per gene: real {real_spread:.5f} | permuted {null_spread:.5f}"
      f" | ratio {real_spread / max(null_spread, 1e-12):.2f}x")
print("(a ratio near 1 means the real matrix carries no more perturbation-specific "
      "structure than the null)")

## 7. What landed on disk

```
data/posterior_matrices/<context>/
    PosteriorMean_matrix_<context>.csv     labels x genes, full + subsampled rows
    row_metadata.csv                       label -> perturbation, variant, n_cells_used
    module_finder_manifest.json            parameters, timings, ashr version
    chunk_*.csv                            ComputeSE.py output (mean_diff, se)
    AshrResult_chunk_*.csv                 ashr output, metadata merged back in
```

The intermediates are kept, so `RunAshrPipeline.sh` skips `ComputeSE.py` on a re-run.
Delete `chunk_*.csv` to force a recompute.

In [ ]:
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file() and not path.name.startswith("chunk"):
        print(f"{path.stat().st_size/1e6:9.2f} MB  {path.relative_to(OUT_DIR)}")